# 09 — Matrix/tensor decompositions for head & layer redundancy: storm vs quiet

Follow-up to notebooks 07 (hidden-state/attention skeleton decomposition) and 08 (input-level
climatological patching). Here we ask a different question: **do different attention heads
(and layers) do redundant work, and does that change when the model is given a real storm vs
a climatologically "blank" quiet case?** Three methods, in increasing sophistication:

1. **Singular value spectrum / effective rank** of the hidden state (plain SVD, extends the
   skeleton-decomposition rank-vs-error sweep from notebook 07 into a full spectrum view).
2. **CKA (Centered Kernel Alignment)** between attention heads' *output representations* —
   standard interpretability tool (Kornblith et al. 2019) for comparing whether two heads
   encode the same information, independent of basis/scale.
3. **Tucker decomposition** of the (heads, query, key) attention *weight* tensor — a stricter,
   numerical test of how many head-mode components are needed to reconstruct the raw attention
   patterns (not just their downstream representation).

All three are run on the same case pair as notebooks 07/08: Bebinca storm (init=2024-09-15,
+24h through the 2024-09-16 landfall) vs a quiet control (init=2024-11-15). Method 1 reuses the
raw hidden-state matrices already cached in notebook 07 (no rerun); methods 2/3 required a
small new hook-based capture (window covering the Bebinca landfall area, block 0/non-shifted,
stages 0 and 1 of the backbone).

## 1. Singular value spectrum / effective rank of the hidden state

![](figures/09_tensor_decomposition/spectrum_storm_vs_quiet.png)


Cumulative energy (fraction of total squared Frobenius norm) vs rank, at +24h, per backbone
stage. **Negative result**: the storm and quiet curves are nearly indistinguishable at every
stage (effective rank for 90% energy: stage 0 ~20-28, stage 1 ~238-278, stage 2 ~359-427,
storm and quiet within a few percent of each other at every stage). If anything, storm's
stage-0 spectrum is *slightly more* concentrated (lower rank) than quiet's, the opposite of the
naive "storm injects more information" hypothesis.

**Conclusion**: the hidden state's effective rank/compressibility is a structural property of
backbone depth, not something driven by the presence of a real physical event. This complements
(and slightly tempers) the notebook 07 finding that skeleton landmarks cluster at the storm
core -- the *location* of informative structure shifts with the storm, but the overall
*amount* of exploitable low-rank structure does not.

## 2. CKA between attention heads (output representations)

![](figures/09_tensor_decomposition/cka_heads_matrix.png)


Mean off-diagonal CKA (higher = heads more redundant/similar to each other):

| | stage 0 | stage 1 |
|---|---|---|
| storm | 0.740 | 0.500 |
| quiet | 0.886 | 0.532 |

**Positive result**: heads are *more redundant* with each other in the quiet case than in the
storm case, at both stages (bigger gap at stage 0). Visually, the quiet stage-0 matrix is
almost uniformly high-similarity (yellow/green), while storm's has more spread. Head 5 (stage
0) is a consistent outlier in *both* cases -- a structural specialization independent of the
storm, not something the storm creates.

**Interpretation**: without a real storm to differentiate, the model's heads converge toward
overlapping/redundant behavior; a real storm appears to pull heads toward more distinct roles
-- consistent with the model needing a richer, less redundant internal representation to
handle the more complex real dynamics.

## 3. Tucker decomposition of the (heads, query, key) attention tensor

![](figures/09_tensor_decomposition/tucker_head_rank.png)


Head-mode rank needed for a given Tucker reconstruction fidelity of the raw attention
*weights* (a stricter numerical test than CKA, which compares representations up to
rotation/scale). At stage 0, the storm curve sits consistently *above* the quiet curve --
more head-rank is needed for the same reconstruction error, confirming the CKA finding with an
independent, stricter method. At stage 1 the two curves nearly overlap -- the effect is
concentrated at the finest resolution stage, not present (or much weaker) at the coarser stage.

Rank needed for <10% relative error: storm stage0 = 8/8 (no compression possible at this
tolerance), quiet stage0 = 7/8; stage1 both = 14/16. Tucker is a much stricter test than CKA
(exact numerical reconstruction of softmax attention values vs correlation of representations),
so the fact that the *direction* of the storm-vs-quiet effect agrees between the two very
different methods is a stronger claim than either alone.

## Summary

- Effective rank of the hidden state (method 1) is essentially unaffected by the storm --
  compressibility is architectural, not event-driven.
- Head redundancy (methods 2 and 3, independent techniques) tells a consistent, opposite story:
  the real storm makes heads *less* redundant with each other (more specialized), at least at
  the finest backbone stage. This is a genuinely new finding, distinct from the notebook 07
  skeleton-landmark result (which was about *where* informative structure sits spatially, not
  about *how many distinct computational roles* the heads are playing).
- This was task 1-3 of a 5-task plan; tasks 4 (attention rollout across all layers) and 5
  (sparse autoencoder / monosemantic features) remain as a follow-up, in increasing order of
  implementation cost and risk (SAE in particular needs much more data than a single storm +
  quiet case pair to reliably produce interpretable features).